# Initialization

In [ ]:
import sys, subprocess, os
from pathlib import Path

venv = Path("../.venv").resolve()
if not venv.exists():
    subprocess.check_call([sys.executable, "-m", "venv", str(venv)])
    subprocess.check_call([str(venv / "bin" / "pip"), "install", "-e", ".."])
    subprocess.check_call([str(venv / "bin" / "pip"), "install", "ipykernel"])
    subprocess.check_call([str(venv / "bin" / "python"), "-m", "ipykernel",
                            "install", "--user", "--name", "taylor-dispersion"])
                            
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pickle
import numdifftools as nd

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
from matplotlib.colors import to_rgb
from matplotlib.gridspec import GridSpec
import seaborn as sns

import fluidfoam

from scipy.interpolate import interp1d, CubicSpline, UnivariateSpline, RegularGridInterpolator
from scipy.integrate import solve_ivp, simpson
from scipy.stats import norm
from scipy.sparse.linalg import factorized
from scipy.sparse import diags, eye

parent_dir = str(Path('..').resolve())
cwd = str(Path('.').resolve())
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from taylor_utils import (
    save_pickle, load_pickle,
    weighted_mean, weighted_variance,
    create_velo_full,
    simulation_rectangular, simulation_circular, simulation_dumbell, estimate_diffusion_timescale,
)
from taylor_utils.rect_psi import Psi_uPrime_avg
from taylor_utils.openfoam import build_openFoam_interp
from taylor_utils.fdm_solver import solve_psi_at_slice, compute_dumbbell_psi_field, compute_dumbbell_psi_interpolator
from taylor_utils.ode_solver import solve_moment_ode, solve_moment_ode_circular
from taylor_utils.pde_solver import solve_concentration_pde
from taylor_utils.inverse_problem import sigma2_const, sigma2_sin_drift, sigma2_sin_nodrift, solve_inverse_problem

output_path = Path('../data_output/')
output_path.mkdir(exist_ok=True)
figures_path = Path('../data_output/figures/')
figures_path.mkdir(parents=True, exist_ok=True)
figures_path = str(figures_path) + '/'
output_path = str(output_path) + '/'

sns.set_style('whitegrid')
plt.rcParams['text.usetex'] = True

NUM_SIMULATIONS = 5

##### This jupyter notebook can be used to re-run all the simulations for the manuscript "Taylor dispersion in long straight channels of arbitrary and gradually changing size and shape". This codebase has only been tested on Ubuntu 24.04.4 LTS.

# Simulations

### Figure 2 Simulation

In [ ]:
func_x = lambda x: 1 + 0.2 * np.sin(x / 400)
func_x_prime = lambda x: 0.2 * np.cos(x / 400) / 400

sim_configs = [
    {"Pe0": 10, "kwargs": {"A": 1500}},
    {"Pe0": 100, "kwargs": {"A": 1500}},
    {"Pe0": 1000, "kwargs": {"A": 15000}},
]

for cfg in sim_configs:
    Pe0 = cfg["Pe0"]
    for seed in range(NUM_SIMULATIONS):
        result = simulation_circular(
            Pe0=Pe0, func_x=func_x, func_x_prime=func_x_prime,
            Nt0=50000/Pe0, seed=seed, sigx2_0=10, sampling_ratio=50, **cfg["kwargs"]
        )
        save_pickle(result, output_path + f"result_periodic2_Pe0_{Pe0}_moment_initVar10_seed_{seed}")
        print(f"Pe0={Pe0}, seed={seed}")

# Summarize results (remove particle positions to save space)
for Pe0 in [10, 100, 1000]:
    name = f"result_periodic2_Pe0_{Pe0}_moment_initVar10_seed_"
    for i in range(NUM_SIMULATIONS):
        result = load_pickle(output_path + name + str(i))
        for key in ["x", "r", "theta"]:
            del result[key]
        save_pickle(result, output_path + name + str(i) + "_summarized")

### Figure 2 PDE Numerical Solution


In [ ]:
func_x = lambda x: 1 + 0.2 * np.sin(x / 400)
U0 = 1
a0 = 1
sigx2_0 = 10
nx = 52000
domain_num = 10
period = 2 * np.pi * 400

def find_U_D_eff(x, D):
    U_func = lambda a: U0 * func_x(0)**2 / a**2
    b_func = lambda x: 0.2 * np.cos(x / 400) / 400
    g_func = lambda x: -0.2 * np.sin(x / 400) / 400 / 400

    a = func_x(x)
    b = b_func(x)
    U = U_func(a)

    U_eff = -b * D * (4 + b**2 - 3 * a * g_func(x)) / 2 / a + U
    D_eff = D * (1 - b**2 - a * b * U / 12 / D + a**2 * U**2 / 48 / D**2)
    return U_eff, D_eff

x_interp = np.linspace(0, period, 5000)
Area_interp = lambda x: func_x(x)**2 * np.pi * a0

pe0_configs = [
    {"Pe0": 10, "max_tau": 5000},
    {"Pe0": 100, "max_tau": 500},
    {"Pe0": 1000, "max_tau": 50},
]
target_taus = [5, 10, 20, 30, 40]

for cfg in pe0_configs:
    Pe0 = cfg["Pe0"]
    max_tau = cfg["max_tau"]
    D = 1.0 / Pe0

    saved_taus = [t for t in target_taus if t <= max_tau]

    name_head = f"result_periodic2_Pe0_{Pe0}_moment_initVar10_seed_"
    result = load_pickle(output_path + name_head + "0_summarized")
    dt = result['dt'] * 10
    t_final = max_tau * a0**2 / D * 1.05
    nt = int(np.ceil(t_final / dt))

    timestamps = [int(round(t * (a0**2 / D) / dt)) for t in saved_taus]

    U_eff_vals, D_eff_vals = find_U_D_eff(x_interp, D)
    U_eff_interp = CubicSpline(x_interp, U_eff_vals)
    D_eff_interp = CubicSpline(x_interp, D_eff_vals)

    moments = solve_concentration_pde(
        U_eff_interp, D_eff_interp, Area_interp,
        dt=dt, nt=nt,
        x_min=-0.5 * period, x_max=(domain_num - 0.5) * period, nx=nx,
        sigx2_0=sigx2_0, num_sub=5000,
        period=period,
        store_timesteps=timestamps,
    )
    save_pickle(moments, output_path + f"con_field_Pe0_{Pe0}_circ")
    print(f"Done: Pe0={Pe0}, max_tau={max_tau}, saved_taus={saved_taus}")

### Figure 4 and Figure 5 Simulation

In [ ]:
case_dir = '/dumbell_geo_short_quarter_re0.1'

Pe0 = 5
name_head = f"result_dumbell_Pe0_{Pe0}_L0_3_moment_initVar10_seed_"

ux_interp, uy_interp, uz_interp = build_openFoam_interp(parent_dir+case_dir, cache_file=output_path + 'foam_data_cache.npz')
save_pickle(ux_interp, output_path + 'ux_dumbellGeometry')
save_pickle(uy_interp, output_path + 'uy_dumbellGeometry')
save_pickle(uz_interp, output_path + 'uz_dumbellGeometry')

L0 = 3
for seed in range(NUM_SIMULATIONS):
    result = simulation_dumbell(
        ux_interp, uy_interp, uz_interp, Pe0, L0,
        scale_factor=10, Nt0=190, seed=0, sigx2_0=10, upper_bound=None, sampling_ratio=5000,
    )
    save_pickle(result, output_path + f"result_dumbell_Pe0_{Pe0}_L0_{L0}_moment_initVar10_seed_{seed}")
    print(f"seed={seed}")

# Summarize results
for i in range(NUM_SIMULATIONS):
    result = load_pickle(output_path + name_head + str(i))
    for key in ["x", "y", "z"]:
        del result[key]
    save_pickle(result, output_path + name_head + str(i) + "_summarized")
result = load_pickle(output_path + name_head + "0_summarized")

D = result['D']
dt = result['dt'] * 100
nt = int(60 * 9 / dt / 0.6 * 1.2)
x_vals_psi = np.linspace(0, 1200, 100)

up_Psi_dict = compute_dumbbell_psi_field(
    x_vals_psi, ux_interp, uy_interp, uz_interp, D
)
save_pickle(up_Psi_dict, output_path + 'up_Psi_Pe0_5')
print('Saved up_Psi_Pe0_5')

psi_interp = compute_dumbbell_psi_interpolator(
    x_vals_psi, ux_interp, uy_interp, uz_interp, D
)
save_pickle({'Psi_interp': psi_interp}, output_path + 'up_Psi_Pe0_5_interp')
print('Saved up_Psi_Pe0_5_interp')

slope = -0.00075
z_func = lambda x: np.where(x < 1200, np.where(x < 0, 1, slope * x + 1), 0.1)

up_Psi_dict = load_pickle(output_path + 'up_Psi_Pe0_5')
H_list = up_Psi_dict['H']
up_Psi_list = up_Psi_dict['up_Psi']
Gamma_list = up_Psi_dict['Gamma']
area_list = up_Psi_dict['area_vals']
avg_ux_list = up_Psi_dict['avg_ux']
avg_uy_list = up_Psi_dict['avg_uy']
avg_uz_list = up_Psi_dict['avg_uz']
avg_dpsi_dx = up_Psi_dict['avg_dpsi_dx']
avg_dpsi_dy = up_Psi_dict['avg_dpsi_dy']
avg_dpsi_dz = up_Psi_dict['avg_dpsi_dz']
avg_up_dpsi_dx = up_Psi_dict['avg_up_dpsi_dx']
avg_up_dpsi_dy = up_Psi_dict['avg_up_dpsi_dy']
avg_up_dpsi_dz = up_Psi_dict['avg_up_dpsi_dz']
avg_d2psi_dx2 = up_Psi_dict['avg_d2psi_dx2']
x_vals_og = up_Psi_dict['x_vals']

U_eff_vals = (avg_ux_list + avg_ux_list * avg_dpsi_dx + avg_uy_list * avg_dpsi_dy
              + avg_uz_list * avg_dpsi_dz + avg_up_dpsi_dx + avg_up_dpsi_dy
              + avg_up_dpsi_dz + D * Gamma_list - D * avg_d2psi_dx2)
D_eff_vals = D * (1 + 2 * avg_dpsi_dx - up_Psi_list / D)

U_eff_interp_full = CubicSpline(x_vals_og, U_eff_vals)
D_eff_interp_full = CubicSpline(x_vals_og, D_eff_vals)
Area_interp_full = CubicSpline(x_vals_og, area_list)

U_eff_interp = lambda x: U_eff_interp_full(np.clip(x, 0, 1200))
D_eff_interp = lambda x: D_eff_interp_full(np.clip(x, 0, 1200))
Area_interp = lambda x: Area_interp_full(np.clip(x, 0, 1200))

T_arr = np.arange(0, nt * dt, dt)
timesteps = [np.abs(T_arr * D / L0**2 - val).argmin() for val in (2, 10, 25, 40, 55)]

moments = solve_concentration_pde(
    U_eff_interp, D_eff_interp, Area_interp,
    dt=dt, nt=nt, x_min=-200, x_max=2000, nx=50000,
    sigx2_0=10.0, num_sub=5000, store_timesteps=timesteps,
)
save_pickle(moments, output_path + f"con_field_Pe0_{int(L0/D)}_dumbell")

### Figure 4 ODE Precomputation


In [ ]:
# Figure 4 ODE solution: dumbbell channel moments
sub_step = 200
slope = -0.00075
z_func = lambda x: np.where(x < 1200, np.where(x < 0, 1, slope * x + 1), 0.1)

up_Psi_dict = load_pickle(output_path + 'up_Psi_Pe0_5')
up_Psi_vals = up_Psi_dict['up_Psi']
H_vals = up_Psi_dict['H']
Gamma_vals = up_Psi_dict['Gamma']
ux_vals = up_Psi_dict['avg_ux']
x_vals = up_Psi_dict['x_vals']

name_head = "result_dumbell_Pe0_5_L0_3_moment_initVar10_seed_"
result = load_pickle(output_path + name_head + "0_summarized")
T = result['T']
D = result['D']

Gamma_interp = interp1d(x_vals, Gamma_vals, kind='cubic', bounds_error=False, fill_value=0.0)
Psi_interp = interp1d(x_vals, up_Psi_vals, kind='cubic', bounds_error=False, fill_value=(up_Psi_vals[0], up_Psi_vals[-1]))
u_interp = interp1d(x_vals, ux_vals, kind='cubic', bounds_error=False, fill_value=(ux_vals[0], ux_vals[-1]))

weighted_x_mean = []
weighted_var_mean = []
for i in range(1):
    res = load_pickle(output_path + name_head + str(i) + "_summarized")
    weighted_x_mean.append(res['weighted_x'].flatten())
    weighted_var_mean.append(res['weighted_var'].flatten())
weighted_x_mean = np.mean(np.vstack(weighted_x_mean), axis=0)
weighted_var_mean = np.mean(np.vstack(weighted_var_mean), axis=0)

closest_index = np.argmin(np.abs(weighted_x_mean - 1250))
predicted_x_bar, predicted_var = solve_moment_ode(
    dt=(T[1] - T[0]) / sub_step, nt=closest_index * sub_step, D=D,
    Gamma_interp=Gamma_interp, u_interp=u_interp, Psi_interp=Psi_interp,
    sigx2_0=10.0)

# Save so the Fig 4 visualization loads these explicitly instead of relying on
# the bare globals (which Fig 7's precompute later overwrites).
save_pickle({'predicted_x_bar': predicted_x_bar, 'predicted_var': predicted_var},
            output_path + 'ode_dumbell_Pe0_5')
print('Saved ode_dumbell_Pe0_5')


### Figure 4 Diffusion Timescale Estimation


In [ ]:
np.random.seed(0)

slope = -0.00075
z_func = lambda x: np.where(x < 1200, np.where(x < 0, 1, slope * x + 1), 0.1)

result = load_pickle(output_path + "result_dumbell_Pe0_5_L0_3_moment_initVar10_seed_0_summarized")
D = result['D']

x_star = np.linspace(0, 1200, 30)
H_array = 2.0 * z_func(x_star)

a0_vals_dict = estimate_diffusion_timescale(
    H_array, D=D, dt=1e-3, N_particles=5000, L=4.0, R=1.0, n_trials=100,
)
save_pickle(a0_vals_dict, output_path + 'a0_dumbell_Pe0_5')
print('Saved a0_dumbell_Pe0_5')


### Figure 7 Simulation

In [ ]:
# Generate inverse problem channel shapes (fx_func) for all three variance targets
name_heads = [
    (sigma2_const, 'constant_variance_inverse_'),
    (sigma2_sin_drift, 'sin_variance_inverse_'),
    (sigma2_sin_nodrift, 'sin_nodrift_variance_inverse_'),
]
for sigma2_func, name_head in name_heads:
    fx_func = solve_inverse_problem(sigma2_func, D=0.1, up_init=1.0, f0=1.0,
                                    x_forward=(0, 900, 450), x_backward=(-150, 0, 75))
    save_pickle(fx_func, output_path + f'fx_func_{name_head}0')
    print(f"Generated fx_func_{name_head}0")

for _, name_head in name_heads:
    fx_func = load_pickle(output_path + f'fx_func_{name_head}0')
    u_interp, v_interp, w_interp = create_velo_full(fx_func, num_x_grid=400, origin=-200, x_max=900)
    save_pickle(u_interp, output_path + f'u_{name_head}0')
    save_pickle(v_interp, output_path + f'v_{name_head}0')
    save_pickle(w_interp, output_path + f'w_{name_head}0')

    for seed in range(NUM_SIMULATIONS):
        result = simulation_rectangular(
            fx_func, 10, u_interp, v_interp, w_interp,
            Nt0=150, seed=seed, sigx2_0=300, upper_bound=900, sampling_ratio=100,
        )
        save_pickle(result, output_path + f"{name_head}{seed}")
        print(f"{name_head} seed={seed}")

    # Summarize results
    for i in range(NUM_SIMULATIONS):
        result = load_pickle(output_path + name_head + str(i))
        for key in ["x", "y", "z"]:
            del result[key]
        save_pickle(result, output_path + name_head + str(i) + "_summarized")

### Figure 7 ODE Precomputation


In [ ]:
# Figure 7 ODE solutions: inverse problem channels
sub_step = 1
num_x = 100

fig7_ode_results = {}
for name_head in ['constant_variance_inverse_', 'sin_variance_inverse_', 'sin_nodrift_variance_inverse_']:
    result = load_pickle(output_path + name_head + '0_summarized')
    func_x = load_pickle(output_path + f"fx_func_{name_head}0")
    u_full_interp = load_pickle(output_path + f"u_{name_head}0")
    x_vals = np.linspace(-100, 900, num_x)
    T = result['T']
    D = result['D']
    Gamma_vals = -1 * func_x.derivative()(x_vals) / func_x(x_vals)
    Psi_u_vals = Psi_uPrime_avg(x_vals, func_x(x_vals), func_x.derivative()(x_vals), D, up_init=1)

    y = np.linspace(-1, 1, 500)
    z = np.linspace(-1, 1, 500)
    X, Y, Z = np.meshgrid(x_vals, y, z, indexing='ij')
    coords = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])
    u_vals = u_full_interp(coords).reshape(X.shape)
    u_mean_vals = np.mean(np.mean(u_vals, axis=2), axis=1)

    Gamma_interp_fig7 = interp1d(x_vals, Gamma_vals, kind='cubic', fill_value='extrapolate')
    Psi_interp_fig7 = interp1d(x_vals, Psi_u_vals, kind='cubic', fill_value='extrapolate')
    u_interp_fig7 = interp1d(x_vals, u_mean_vals, kind='cubic', fill_value='extrapolate')

    predicted_x_bar, predicted_var = solve_moment_ode(
        dt=(T[1] - T[0]) / sub_step, nt=T.size * sub_step, D=D,
        Gamma_interp=Gamma_interp_fig7, u_interp=u_interp_fig7, Psi_interp=Psi_interp_fig7,
        sigx2_0=300.0)

    fig7_ode_results[name_head] = {
        'predicted_x_bar': predicted_x_bar,
        'predicted_var': predicted_var,
    }
    print(f"Done: {name_head}")

save_pickle(fig7_ode_results, output_path + 'fig7_ode_results')


# Visualizations

In [ ]:
# Shared visualization setup
mpl.rcParams.update(mpl.rcParamsDefault)
sns.set_context('talk')
sns.set_style('ticks', {'axes.grid': False})
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Times New Roman'],
    'font.weight': 'normal',
    'axes.unicode_minus': False,
    'text.latex.preamble': r"""
        \usepackage{mathptmx}
    """,
})

C1 = mcolors.to_rgb('C1')
DC1 = tuple([x * 0.9 for x in C1])


def load_seed_averages(name_head, n_seeds=NUM_SIMULATIONS):
    """Load simulation results across seeds and compute mean trajectory."""
    weighted_x = []
    weighted_var = []
    for i in range(n_seeds):
        result = load_pickle(output_path + name_head + str(i) + "_summarized")
        weighted_x.append(result['weighted_x'].flatten())
        weighted_var.append(result['weighted_var'].flatten())
    return np.mean(np.vstack(weighted_x), axis=0), np.mean(np.vstack(weighted_var), axis=0)

### Figure 2 Visualization

In [ ]:
func_x = lambda x: 1 + 0.2 * np.sin(x / 400)
func_x_circ = func_x
a0 = 1
U0 = 1

fig = plt.figure(figsize=(15, 8))
gs_main = fig.add_gridspec(4, 1, figure=fig, hspace=0.8, wspace=0)

# Panel (a): particle scatter + concentration for Pe0=1000
ax1000 = fig.add_subplot(gs_main[0])
name_head = "result_periodic2_Pe0_1000_moment_initVar10_seed_"
result = load_pickle(output_path + name_head + "0")
num_result = load_pickle(output_path + "con_field_Pe0_1000_circ")
stored_concentration = num_result['con_field']

x_vals = np.linspace(0, 45000, 1000)
ax1000.plot(x_vals, func_x(x_vals), color='black')
ax1000.plot(x_vals, -func_x(x_vals), color='black')

T = result['T']
D = result['D']
target_taus = [5, 10, 20, 30, 40]
tau = T / (a0**2 / D)
j_range = [int(np.argmin(np.abs(tau - t))) for t in target_taus]
for m, j2 in enumerate(j_range):
    estimated_loc = result['weighted_x'].flatten()[j2]
    X = result['x'][:, j2]
    R = np.copy(result['r'][:, j2])
    THETA = result['theta'][:, j2]
    loc_neg = THETA < 0
    R_plot = R[~loc_neg]
    Y = R_plot * np.sin(THETA[~loc_neg])
    X_plot = X[~loc_neg]
    ax1000.scatter(X_plot, Y, s=1, alpha=0.5)

    if m == 0:
        ax1000.text(estimated_loc - 1000, 2.5, r'$tD/a_o^2=$', ha='right', va='center', fontsize=25)
        ax1000.text(estimated_loc, 1.34, fr'${{{int(T[j2] / (a0**2 / D))}}}$', ha='center', va='bottom', fontsize=22)
    else:
        ax1000.text(estimated_loc, 1.34, fr"${int(T[j2] / (a0**2 / D))}$", ha='center', va='bottom', fontsize=22)

    xmin, xmax = np.min(X_plot), np.max(X_plot)
    i_plot = 1000
    x_pdf = np.linspace(xmin - 0.5 * (xmax - xmin), xmax + 0.5 * (xmax - xmin), i_plot)

    alpha_all = np.maximum(stored_concentration[m], 0)
    alpha_all = alpha_all / np.max(alpha_all)
    # use each snapshot's absolute (lab-frame) coords: with co-moving
    # shifting the stored field is rolled, so its true x is con_x_locations[m]
    analytical_x_vals = (num_result['con_x_locations'][m]
                         if num_result.get('con_x_locations') else num_result['x_locations'])
    q = np.nanargmin(np.abs(x_pdf[1] - analytical_x_vals))
    q_plus = (x_pdf[1] - x_pdf[0]) / (analytical_x_vals[1] - analytical_x_vals[0])

    for i in range(i_plot - 1):
        q_int = round(q)
        if q_int >= 52000:
            continue
        ax1000.fill_between(x_pdf[i:i+2], -func_x(x_pdf[i:i+2]), color=f'C{m}', alpha=alpha_all[q_int])
        q += q_plus

ax1000.tick_params(axis='both', direction='in', labelsize=22, length=7, width=2)
ax1000.set_xlim([-1000, 45000])
ax1000.set_xlabel(r'$x^*$', fontsize=25)
ax1000.set_ylabel(r'$r^*$', fontsize=25)
ax1000.set_xticks([0, 10000, 20000, 30000, 40000])
ax1000.set_yticks([-1, 1])

import gc
del result, num_result, stored_concentration
gc.collect()

# Panels (b)-(d): variance for Pe0=10, 100, 1000
sub_step = 1
pe_configs = [
    {"Pe0": 10,   "D": 0.1,   "xlim": [-1000, 45000], "xticks": [0, 10000, 20000, 30000, 40000],
     "yticks": None, "pe_label_x": 25000, "pe_label_y": 120000, "show_legend": True, "ylim": [-10000, 150000]},
    {"Pe0": 100,  "D": 0.01,  "xlim": [-1000, 45000], "xticks": [0, 10000, 20000, 30000, 40000],
     "yticks": None, "pe_label_x": 0, "pe_label_y": 450000, "show_legend": False, "ylim": None},
    {"Pe0": 1000, "D": 0.001, "xlim": [-1000, 45000], "xticks": [0, 10000, 20000, 30000, 40000],
     "yticks": None, "pe_label_x": 0, "pe_label_y": 4500000, "show_legend": False, "ylim": None,}
]

variance_axes = []
for idx, cfg in enumerate(pe_configs):
    ax = fig.add_subplot(gs_main[idx + 1])
    variance_axes.append(ax)
    Pe0 = cfg["Pe0"]
    D_val = cfg["D"]
    name_head = f"result_periodic2_Pe0_{Pe0}_moment_initVar10_seed_"
    num_result = load_pickle(output_path + f"con_field_Pe0_{Pe0}_circ")
    weighted_x_mean, weighted_var_mean = load_seed_averages(name_head)

    result = load_pickle(output_path + name_head + "0_summarized")
    T = result['T']
    predicted_mean_ = num_result['predicted_mean']
    predicted_variance_ = num_result['predicted_variance']
    del result, num_result
    gc.collect()

    ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
    ax.plot(predicted_mean_, predicted_variance_,
            label=r'$\mathrm{PDE} \hspace{0.25em} \mathrm{Model}$', color='C2', alpha=1)

    predicted_x_bar_approx, predicted_var_approx = solve_moment_ode_circular(
        dt=(T[1] - T[0]) / sub_step, nt=T.size * sub_step, D=D_val,
        func_x=func_x_circ, sigx2_0=10.0)
    ax.plot(predicted_x_bar_approx, predicted_var_approx,
            label=r'$\mathrm{ODE} \hspace{0.25em} \mathrm{Model}$',
            color=DC1, alpha=1, linestyle='dashed')
    ax.plot(weighted_x_mean, weighted_var_mean,
            label=r'$\mathrm{Brownian}$', color='C0', linestyle='dotted')

    ax.set_xlabel(r'$\overline{x}^*$', fontsize=25)
    ax.set_ylabel(r'$\sigma^{*2}_x$', fontsize=25)
    ax.set_xlim(cfg["xlim"])
    ax.set_xticks(cfg["xticks"])
    ax.tick_params(axis='both', direction='in', length=7, width=2, labelsize=22)

    if cfg["ylim"] is not None:
        ax.set_ylim(cfg["ylim"])
    if cfg["yticks"] is not None:
        ax.set_yticks(cfg["yticks"])
    if cfg["show_legend"]:
        ax.legend(loc='upper left', ncols=3, fontsize=18, columnspacing=1)

    ax.text(cfg["pe_label_x"], cfg["pe_label_y"], fr'$Pe_a = {Pe0}$', color='black', fontsize=22,
            horizontalalignment='left', verticalalignment='top')

ax1000.text(31300, 2.5, r'$Pe_a = 1000$', color='black', fontsize=22, horizontalalignment='left', verticalalignment='center')

# Panel labels
ax1000.text(-0.06, 1.1, r'$(a)$', transform=ax1000.transAxes, fontsize=22, va='top', ha='left')
for i, label in enumerate([r'$(b)$', r'$(c)$', r'$(d)$']):
    variance_axes[i].text(-0.06, 1.25, label, transform=variance_axes[i].transAxes, fontsize=22, va='top', ha='left')

plt.show()


### Figure 3(b) Visualizations

Figure 3(a) was visualized using ParaView:

Ahrens, James, Geveci, Berk, Law, Charles, ParaView: An End-User Tool for Large Data Visualization, Visualization Handbook, Elsevier, 2005, ISBN-13: 9780123875822.

In [ ]:
# Load velocity interpolators
ux_interp = load_pickle(output_path + 'ux_dumbellGeometry')
uy_interp = load_pickle(output_path + 'uy_dumbellGeometry')
uz_interp = load_pickle(output_path + 'uz_dumbellGeometry')

x_vals = np.linspace(0, 1200, 3)
plot_data = []
D = 0.6
z_func = lambda x: np.where(x < 1200, np.where(x < 0, 1, -0.00075 * x + 1), 0.1)

for i in range(x_vals.size):
    res = solve_psi_at_slice(x_vals[i], ux_interp, uy_interp, uz_interp, D, z_func=z_func)
    y = res['y']
    z = res['z']
    Y, Z = np.meshgrid(y, z, indexing='xy')
    plot_data.append({'Y': Y, 'Z': Z, 'psi': res['psi'], 'x_val': x_vals[i]})

all_psi = [d['psi'] for d in plot_data]
v_min = np.min([np.ma.min(p) for p in all_psi])
v_max = np.max([np.ma.max(p) for p in all_psi])
levels = np.linspace(v_min, v_max, 200)

border_width = 1
fig, axes = plt.subplots(1, 3, figsize=(7, 4), dpi=300)
for i, (ax, data) in enumerate(zip(axes, plot_data)):
    Y_grid = data['Y']
    Z_grid = data['Z']
    psi_field = data['psi']

    plot = ax.contourf(Y_grid, Z_grid, psi_field, levels=levels,
                       cmap='viridis', vmin=v_min, vmax=v_max)

    for spine in ax.spines.values():
        spine.set_linewidth(border_width)
    plot.set_edgecolor("face")

    ax.tick_params(direction='in', which='both', length=3, width=border_width, labelsize=15)
    ax.set_xticks([-3, 3])
    ax.set_xlim([-3.2, 3.2])
    ax.set_yticks([-1, 1])
    ax.set_ylim([-1.2, 1.2])
    ax.set_aspect('equal')
    ax.set_xlabel(r'$y^*$', fontsize=17)
    ax.xaxis.set_label_coords(0.5, -0.15)
    ax.set_title(rf'$\Psi(x^* = {int(x_vals[i])})$', fontsize=17, y=1.01)
    if i == 0:
        ax.set_ylabel(r'$z^*$', fontsize=17)
        ax.yaxis.set_label_coords(-0.06, 0.5)
    else:
        ax.tick_params(labelleft=False)

cbar = fig.colorbar(plot, ax=axes, location='right', fraction=0.03, aspect=4, pad=-2)
cbar.outline.set_linewidth(border_width)
cbar.set_ticks([-0.4, 0.6, 1.6])
cbar.ax.tick_params(direction='out', labelsize=16, length=7, width=border_width)
plt.subplots_adjust(left=0.03, right=0.87, top=0.87, bottom=0.13, wspace=0.11)

plt.show()

### Figure 4 Visualization

In [ ]:
# Load data needed from simulation cells
pde_result = load_pickle(output_path + 'con_field_Pe0_5_dumbell')
name_head = "result_dumbell_Pe0_5_L0_3_moment_initVar10_seed_"
result = load_pickle(output_path + name_head + "0")
T = result['T']
D = result['D']
ode_res = load_pickle(output_path + 'ode_dumbell_Pe0_5')
predicted_x_bar = ode_res['predicted_x_bar']
predicted_var = ode_res['predicted_var']
j_range = [np.abs(T * D / 9 - val).argmin() for val in (2, 10, 25, 40, 55)]

fig = plt.figure(figsize=(13, 8.1), constrained_layout=False)
gs_main = fig.add_gridspec(3, 1, figure=fig, hspace=0.45, wspace=0)
num_x = 200
ax1 = fig.add_subplot(gs_main[0])
x_vals_display = np.linspace(-25, 1225, num_x)
slope = -0.00075
z_func = lambda x: np.where(x < 1200, np.where(x < 0, 1, slope * x + 1), 0.1)
ax1.plot(x_vals_display, z_func(x_vals_display), color='black')
ax1.plot(x_vals_display, -z_func(x_vals_display), color='black')

i_plot = 700
y_plot = 250
z_plot = 600
c_list = pde_result['con_field']
c_x_list = pde_result['con_diff']
x_vals_pde = pde_result['x_locations']

interp_dict = load_pickle(output_path + 'up_Psi_Pe0_5_interp')
psi_interp = interp_dict['Psi_interp']
for m, j2 in enumerate(j_range):
    color_code = f'C{m}'
    rgb_color = mcolors.to_rgb(color_code)
    cmap_custom = mcolors.LinearSegmentedColormap.from_list(
        'my_custom_cmap', [(1, 1, 1), rgb_color], N=100)

    c_interp = interp1d(x_vals_pde, c_list[m], kind='linear', bounds_error=False, fill_value=0.0)
    c_x_interp = interp1d(x_vals_pde, c_x_list[m], kind='linear', bounds_error=False, fill_value=0.0)

    estimated_loc = result['weighted_x'].flatten()[j2]

    X = result['x'][:, j2]
    Z = np.copy(result['z'][:, j2])
    loc_neg = Z < 0
    Z_plot = Z[~loc_neg]
    X_plot = X[~loc_neg]
    loc_over = Z_plot > z_func(X_plot)
    Z_plot_sub = Z_plot[~loc_over]
    X_plot_sub = X_plot[~loc_over]
    ax1.scatter(X_plot_sub, Z_plot_sub, s=1, alpha=0.5)

    if m == 0:
        ax1.text(estimated_loc - 95, 1.47, r'$tD/l_o^2=$', ha='center', va='bottom', fontsize=25)
        ax1.text(estimated_loc, 1.22, fr'${round(T[j2] * D / 9)}$', ha='center', va='bottom', fontsize=22)
    else:
        ax1.text(estimated_loc, 1.22, fr'${round(T[j2] * D / 9)}$', ha='center', va='bottom', fontsize=22)

    x_bar = estimated_loc
    x_pdf = np.linspace(x_bar - 75 * (m + 1), x_bar + 75 * (m + 1), i_plot)
    y_pdf = np.linspace(-3, 3, y_plot)
    z_pdf = np.linspace(0, -z_func(x_bar - 75 * (m + 1)), z_plot)

    X, Y, Z = np.meshgrid(x_pdf, y_pdf, z_pdf)
    pts = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])
    psi_vals = psi_interp(pts)
    psi_grid = psi_vals.reshape(X.shape)

    c_vals = c_interp(x_pdf)
    c_x_vals = c_x_interp(x_pdf)
    c_vals_3d = c_vals.reshape(1, i_plot, 1) / np.sum(c_vals)
    c_x_vals_3d = c_x_vals.reshape(1, i_plot, 1) / np.sum(c_vals)

    c_grid = c_vals_3d + c_x_vals_3d * psi_grid
    sum_c = np.sum(c_grid, axis=0)
    normed_c = np.clip(sum_c / np.max(sum_c) / 0.4, 0, 1)
    normed_c[normed_c < 0.001] = np.nan

    X_2d, Z_2d = np.meshgrid(x_pdf, z_pdf)
    z_limit = z_func(X_2d)
    c_masked = normed_c.T.copy()
    c_masked[Z_2d < -z_limit] = np.nan

    ax1.pcolormesh(X_2d, Z_2d, c_masked, shading='auto', cmap=cmap_custom, zorder=0)

ax1.tick_params(axis='both', direction='in', labelsize=22)
ax1.set_xlabel(r'$x^*$', fontsize=25)
ax1.set_ylabel(r'$z^*$', fontsize=25)
ax1.set_xlim([-50, 1250])
ax1.set_ylim([-1.2, 1.2])
ax1.set_yticks([-1, 1])

# Panel (b): variance
weighted_x_mean, weighted_var_mean = load_seed_averages(name_head)

ax2 = fig.add_subplot(gs_main[1])
ax2.plot(predicted_x_bar, predicted_var, label=r'$\mathrm{ODE} \hspace{0.25em} \mathrm{model}$', color='C2', alpha=1)
ax2.plot(pde_result['predicted_mean'].ravel(), pde_result['predicted_variance'].ravel(),
         label=r'$\mathrm{PDE} \hspace{0.25em} \mathrm{model}$', color=DC1, alpha=1, linestyle='dashed')
ax2.plot(weighted_x_mean, weighted_var_mean, label=r'$\mathrm{Brownian}$', color='C0', linestyle='dotted', zorder=100)
ax2.set_xlabel(r'$\overline{x}^*$', fontsize=25)
ax2.set_ylabel(r'$\sigma^{*2}_x$', fontsize=25)
ax2.set_xlim([-50, 1250])
ax2.tick_params(axis='both', direction='in', labelsize=20)
ax2.legend(loc='upper left', bbox_to_anchor=[0.01, 0.998], fontsize=22, ncols=3, columnspacing=0.4)

# Panel (c): smallness parameter + Peclet number
ax3 = fig.add_subplot(gs_main[2])
a0_vals_dict = load_pickle(output_path + 'a0_dumbell_Pe0_5')
a0_vals = np.sqrt(a0_vals_dict['time_results'] * D)
H_vals = a0_vals_dict['H_values']
x_values = (H_vals / 2 - 1) / slope
a0_interp = UnivariateSpline(x_values, a0_vals, s=0.2)

observed_var_interp = interp1d(weighted_x_mean, weighted_var_mean, kind='cubic')

x_interp = np.linspace(0, 1200, 100)
smallness_parameter = a0_interp(x_interp)**2 * u_interp(x_interp) / D / (4 * np.sqrt(observed_var_interp(x_interp)))
sigma_lambda = 4 * np.sqrt(observed_var_interp(x_interp)) / 1200
peclet_number = a0_interp(x_interp) * u_interp(x_interp) / D

line1 = ax3.plot(x_interp, smallness_parameter, label=r'$\eta Pe$', color='C0', linestyle='dotted')
ax3.set_ylabel(r'$\eta Pe$', color='C0', fontsize=25)
ax3.tick_params(axis='x', colors='black', labelsize=22, direction='in')
ax3.tick_params(axis='y', colors='C0', labelsize=22, direction='in')
ax3.set_ylim([0.25, 1.75])
ax3.spines['left'].set_color('C0')
ax3.spines['right'].set_visible(False)
ax3.set_xlabel(r'$\overline{x}^*$', fontsize=25)

ax4 = ax3.twinx()
line2 = ax4.plot(x_interp, peclet_number, label=r'$Pe$', color='C2')
ax4.set_ylabel(r'$Pe$', color='C2', fontsize=25)
ax4.tick_params(axis='y', colors='C2', labelsize=22, direction='in')
ax4.set_ylim([4, 31])
ax4.spines['right'].set_color('C2')
ax4.spines['left'].set_visible(False)

ax5 = ax3.twinx()
line3 = ax5.plot(x_interp, sigma_lambda, label=r'$\gamma$', color=DC1, linestyle='dashed')
ax5.set_ylabel(r'$\gamma$', color=DC1, fontsize=25)
ax5.tick_params(axis='y', colors=DC1, labelsize=22, direction='in')
ax5.spines['right'].set_color(DC1)
ax5.spines['left'].set_visible(False)
ax5.spines['top'].set_visible(False)
ax5.spines['bottom'].set_visible(False)
ax5.spines['right'].set_position(('outward', 60))

lns = line2 + line3 + line1
labs = [l.get_label() for l in lns]
ax3.legend(lns, labs, loc='upper left', bbox_to_anchor=[0.04, 1.01], fontsize=22, ncols=3, columnspacing=0.4)
ax3.set_xlim([-50, 1250])

ax1.text(-0.1, 1.01, r'$(a)$', transform=ax1.transAxes, fontsize=22, va='top', ha='left')
ax2.text(-0.1, 1.01, r'$(b)$', transform=ax2.transAxes, fontsize=22, va='top', ha='left')
ax3.text(-0.1, 1.01, r'$(c)$', transform=ax3.transAxes, fontsize=22, va='top', ha='left')

plt.show()

### Figure 5 Visualization

In [ ]:
sub_step = 1

slope = -0.00075
z_func = lambda x: np.where(x < 1200, np.where(x < 0, 1, slope * x + 1), 0.1)
y_func = lambda x: 2 - np.sqrt(1 - z_func(x)**2)
L, R = 4.0, 1.0

interp_dict = load_pickle(output_path + 'up_Psi_Pe0_5_interp')
psi_interp = interp_dict['Psi_interp']
fig = plt.figure(figsize=(12, 6), constrained_layout=False)
gs_main = fig.add_gridspec(3, 1, figure=fig, hspace=0.5, wspace=0)

num_x = 200
ax1 = fig.add_subplot(gs_main[0])
pde_result = load_pickle(output_path + 'con_field_Pe0_5_dumbell')
name_head = "result_dumbell_Pe0_5_L0_3_moment_initVar10_seed_"
result = load_pickle(output_path + name_head + "0")
x_vals_display = np.linspace(-25, 1225, num_x)

ax1.plot(x_vals_display, np.ones(num_x) * 3, color='black')
ax1.plot(x_vals_display, np.ones(num_x) * -3, color='black')
ax1.plot(x_vals_display, y_func(x_vals_display), color='black', linestyle='dashed', alpha=0.75, linewidth=1)
ax1.plot(x_vals_display, -y_func(x_vals_display), color='black', linestyle='dashed', alpha=0.75, linewidth=1)

T = result['T']
D = result['D']
j_range = [np.abs(T * D / 9 - val).argmin() for val in (2, 10, 25, 40, 55)]

i_plot = 200
y_plot = 200
z_plot = 200
c_list = pde_result['con_field']
c_x_list = pde_result['con_diff']
x_vals_pde = pde_result['x_locations']

for m, j2 in enumerate(j_range):
    c_interp = interp1d(x_vals_pde, c_list[m], kind='linear')
    c_x_interp = interp1d(x_vals_pde, c_x_list[m], kind='linear')

    estimated_loc = result['weighted_x'].flatten()[j2]

    X = result['x'][:, j2]
    Y = np.copy(result['y'][:, j2])
    Z = np.copy(result['z'][:, j2])
    loc_neg = Y > 0
    Y_plot = Y[loc_neg]
    X_plot = X[loc_neg]

    ax1.scatter(X_plot, Y_plot, s=1, alpha=0.5)
    if m == 0:
        ax1.text(estimated_loc - 90, 4.92, r'$tD/l_o^2=$', ha='center', va='bottom', fontsize=25)
        ax1.text(estimated_loc, 3.52, fr'${round(T[j2] * D / 9)}$', ha='center', va='bottom', fontsize=22)
    else:
        ax1.text(estimated_loc, 3.65, fr'${round(T[j2] * D / 9)}$', ha='center', va='bottom', fontsize=22)

    x_bar = estimated_loc
    x_pdf = np.linspace(x_bar - 50 * (m + 1), x_bar + 50 * (m + 1), i_plot)
    y_pdf = np.linspace(0, -3, y_plot)
    z_pdf = np.linspace(-1, 1, z_plot)

    X, Y, Z = np.meshgrid(x_pdf, y_pdf, z_pdf)
    pts = np.column_stack([X.ravel(), Y.ravel(), Z.ravel()])
    psi_vals = psi_interp(pts)
    psi_grid = psi_vals.reshape(X.shape)

    c_vals = c_interp(x_pdf)
    c_x_vals = c_x_interp(x_pdf)
    c_vals_3d = c_vals.reshape(1, i_plot, 1) / np.sum(c_vals)
    c_x_vals_3d = c_x_vals.reshape(1, i_plot, 1) / np.sum(c_vals)

    c_grid = c_vals_3d * (psi_grid != 0.0) + c_x_vals_3d * psi_grid
    sum_c = np.sum(c_grid, axis=2)
    normed_c = sum_c / np.max(sum_c) / 0.28

    rgba_image = np.zeros((y_plot, i_plot, 4))
    base_rgb = to_rgb(f'C{m}')
    rgba_image[:, :, 0] = base_rgb[0]
    rgba_image[:, :, 1] = base_rgb[1]
    rgba_image[:, :, 2] = base_rgb[2]
    rgba_image[:, :, 3] = normed_c
    extent = [x_pdf[0], x_pdf[-1], -3, 0]
    ax1.imshow(rgba_image, aspect='auto', extent=extent, origin='upper', zorder=0, interpolation='bilinear')

    if m == 2:
        ax2 = fig.add_subplot(gs_main[1])
        ax2.text(355, 3.72, rf'$tD/l_o^2={round(T[j2] * D / 9)}$', ha='center', va='bottom', fontsize=25)
        ax2.scatter(X_plot, Y_plot, s=1, alpha=0.5, color=f'C{m}')
        ax2.imshow(rgba_image, aspect='auto', extent=extent, origin='upper', zorder=0, interpolation='bilinear')

    if m == 4:
        ax3 = fig.add_subplot(gs_main[2])
        ax3.text(937, 3.72, rf'$tD/l_o^2={round(T[j2] * D / 9)}$', ha='center', va='bottom', fontsize=25)
        ax3.scatter(X_plot, Y_plot, s=1, alpha=0.5, color=f'C{m}')
        ax3.imshow(rgba_image, aspect='auto', extent=extent, origin='upper', zorder=0, interpolation='bilinear')

ax1.tick_params(axis='both', direction='in', labelsize=22)
ax1.set_xlabel(r'$x^*$', fontsize=25, labelpad=-15)
ax1.set_ylabel(r'$y^*$', fontsize=25)
ax1.set_xlim([-50, 1250])
ax1.set_ylim([-3.5, 3.5])
ax1.set_yticks([-3, 0, 3])
ax1.set_xticks([0, 400, 800, 1200])

ax2.tick_params(axis='both', direction='in', labelsize=22)
ax2.set_yticks([-3, 0, 3])
ax2.set_ylim(-3.5, 3.5)
ax2.set_xlabel(r'$x^*$', fontsize=25, labelpad=-15)
ax2.set_ylabel(r'$y^*$', fontsize=25)
ax2.set_xlim([315, 510])
ax2.set_xticks([325, 400, 475])

ax3.tick_params(axis='both', direction='in', labelsize=22)
ax3.set_yticks([-3, 0, 3])
ax3.set_ylim(-3.5, 3.5)
ax3.set_xlabel(r'$x^*$', fontsize=25, labelpad=-15)
ax3.set_ylabel(r'$y^*$', fontsize=25)
ax3.set_xlim([875, 1175])
ax3.set_xticks([900, 1000, 1100])

x_vals_display = np.linspace(320, 507, num_x)
ax2.plot(x_vals_display, np.ones(num_x) * 3, color='black')
ax2.plot(x_vals_display, np.ones(num_x) * -3, color='black')
ax2.plot(x_vals_display, y_func(x_vals_display), color='black', linestyle='dashed', alpha=0.75, linewidth=1)
ax2.plot(x_vals_display, -y_func(x_vals_display), color='black', linestyle='dashed', alpha=0.75, linewidth=1)

x_vals_display = np.linspace(882, 1170, num_x)
ax3.plot(x_vals_display, np.ones(num_x) * 3, color='black')
ax3.plot(x_vals_display, np.ones(num_x) * -3, color='black')
ax3.plot(x_vals_display, y_func(x_vals_display), color='black', linestyle='dashed', alpha=0.75, linewidth=1)
ax3.plot(x_vals_display, -y_func(x_vals_display), color='black', linestyle='dashed', alpha=0.75, linewidth=1)

ax1.text(-0.1, 1.01, r'$(a)$', transform=ax1.transAxes, fontsize=22, va='top', ha='left')
ax2.text(-0.1, 1.01, r'$(b)$', transform=ax2.transAxes, fontsize=22, va='top', ha='left')
ax3.text(-0.1, 1.01, r'$(c)$', transform=ax3.transAxes, fontsize=22, va='top', ha='left')

plt.show()

### Figure 6 Visualization

In [ ]:
fx = np.linspace(1, 10, 1000)
fprime = np.linspace(0.001, 0.12, 1000)
# Z is built in 'ij' layout [fx, fprime] (reshaped to FX.shape), so build X/Y the same way.
X, Y = np.meshgrid(fx, fprime, indexing='ij')   # X = half-width, Y = slope

fig = plt.figure(dpi=400, figsize=(10, 12))

pe_values = [5, 50, 100]
for idx, Pe in enumerate(pe_values):
    D = 1.0 / Pe

    FX, FP = np.meshgrid(fx, fprime, indexing='ij')
    psi = Psi_uPrime_avg(FX.ravel(), FX.ravel(), FP.ravel(), D, up_init=1).reshape(FX.shape)
    D_eff = 1 - psi / D 
    Z = np.log(D_eff / Pe / fprime / 4)

    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')
    ax.plot_surface(Y, X, Z, cmap='viridis', edgecolor='none', linewidth=0, antialiased=True, shade=False)
    # x-axis -> slope f'(x)
    ax.set_xticks([0, 0.05, 0.1])
    ax.set_xlim3d([0.0, 0.125])
    # y-axis -> half-width f(x)
    ax.set_yticks([1, 5, 9])
    ax.set_ylim3d([1, 10.5])
    ax.set_zlim([-2, 6])
    ax.set_zticks([-2, 2, 6])
    ax.set_box_aspect(aspect=(1.5, 1, 1))
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.tick_params(axis='y', pad=-4)
    ax.tick_params(axis='z', pad=0)
    ax.tick_params(axis='x', pad=-2)
    ax.text2D(0.2, 0, r"$f' z_0$", transform=ax.transAxes, ha='center', fontsize=15)
    ax.text2D(0.98, 0.07, r"$f(x)$",   transform=ax.transAxes, ha='center', fontsize=15)

plt.show()


### Figure 7 Visualization

In [ ]:
sub_step = 1
fig7_ode_results = load_pickle(output_path + 'fig7_ode_results')

def plot_particle_scatter(ax, name_head, func_x, predicted_x_bar, predicted_var,
                          j_range, shift=0, time_label=r'$tD/\mathrm{z}_o^2=$',
                          label_y_offset=3.93, label_y_time=3.53, first_label_x_offset=-100):
    """Plot particle scatter with concentration fill for Figure 7 panels."""
    result = load_pickle(output_path + name_head + '0')
    T = result['T']
    D = result['D']
    a0 = 1
    for m, j2 in enumerate(j_range):
        estimated_loc = result['weighted_x'].flatten()[j2]
        X = result['x'][:, j2]
        Y = np.copy(result['y'][:, j2])
        loc_neg = Y < 0
        Y_plot = Y[~loc_neg]
        X_plot = X[~loc_neg]
        ax.scatter(X_plot, Y_plot + shift, s=1, alpha=0.5, color=f'C{m}')

        if shift == 0:
            if m == 0:
                ax.text(estimated_loc + first_label_x_offset, label_y_offset,
                        time_label, ha='center', va='bottom', fontsize=25)
                ax.text(estimated_loc, label_y_time,
                        fr'${round(T[j2] / (a0**2 / D))}$', ha='center', va='bottom', fontsize=22)
            else:
                ax.text(estimated_loc, label_y_time,
                        fr'${int(T[j2] / (a0**2 / D))}$', ha='center', va='bottom', fontsize=22)

        std = np.sqrt(predicted_var[j2 * sub_step])
        x_bar = predicted_x_bar[j2 * sub_step]
        i_plot = 200
        x_pdf = np.linspace(x_bar - 3.5 * std, x_bar + 3.5 * std, i_plot)
        alpha_all = norm.pdf(x_pdf, x_bar, std) / np.max(norm.pdf(x_pdf, x_bar, std))
        fill_y0 = shift if shift != 0 else 0
        for i in range(i_plot - 1):
            ax.fill_between(x_pdf[i:i+2], fill_y0, -func_x(x_pdf[i:i+2]) + shift, color=f'C{m}', alpha=alpha_all[i])


    del result
    import gc; gc.collect()


fig = plt.figure(figsize=(15, 10), constrained_layout=False)
gs_main = fig.add_gridspec(2, 2, figure=fig, hspace=0.28, wspace=0.20, height_ratios=[1, 0.65])
num_x = 100

# --- Panel (a): constant variance ---
axTL = fig.add_subplot(gs_main[0])
name_head = 'constant_variance_inverse_'
result = load_pickle(output_path + name_head + '0_summarized')
func_x = load_pickle(output_path + f"fx_func_{name_head}0")
ode_res = fig7_ode_results[name_head]
predicted_x_bar = ode_res['predicted_x_bar']
predicted_var = ode_res['predicted_var']
x_vals_display = np.linspace(-25, 625, num_x)
axTL.plot(x_vals_display, func_x(x_vals_display), color='black')
axTL.plot(x_vals_display, -func_x(x_vals_display), color='black')
T = result['T']
D = result['D']
j_range = [int(np.where(T * D == val)[0][0]) for val in (5, 20, 40, 60, 85)]

plot_particle_scatter(axTL, name_head, func_x, predicted_x_bar, predicted_var, j_range)

axTL.tick_params(axis='both', direction='in', labelsize=22)
axTL.set_xlabel(r'$x^*$', fontsize=25)
axTL.set_ylabel(r'$y^*$', fontsize=25)
axTL.set_xlim([-50, 675])
axTL.set_ylim([-3.5, 3.5])
axTL.set_yticks([-3, -2, -1, 0, 1, 2, 3])

weighted_x_mean, weighted_var_mean = load_seed_averages(name_head)

axBL = fig.add_subplot(gs_main[2])
axBL.plot(weighted_x_mean, weighted_var_mean,
          label=r'$\sigma^{*2}_{x, \mathrm{B}}: \hspace{0.25em} \mathrm{Brownian}$', color='C0', linestyle='dotted')
axBL.plot(predicted_x_bar, predicted_var,
          label=r'$\sigma^{*2}_{x, \mathrm{ODE}}: \hspace{0.25em} \mathrm{ODE} \hspace{0.25em} \mathrm{model}$', color='C2', alpha=1)
x_vals2 = np.linspace(0, int(np.max(weighted_x_mean)), 500)
axBL.plot(x_vals2, sigma2_const(x_vals2),
          label=r'$\sigma^{*2}_{x, \mathrm{target}}: \hspace{0.25em} \mathrm{target}$', color=DC1, alpha=1, linestyle='dashed')
axBL.set_xlabel(r'$\overline{x}^*$', fontsize=25)
axBL.set_ylabel(r'$\sigma^{*2}_x$', fontsize=25)
axBL.set_ylim([250, 500])
axBL.set_xlim([-50, 675])
axBL.tick_params(axis='both', direction='in', labelsize=22)
axBL.legend(loc='upper left', bbox_to_anchor=[0.01, 0.998], fontsize=22)

# --- Panel (b): sin_drift (top) + sin_nodrift (bottom) ---
shift_fac = 4.8
axTR = fig.add_subplot(gs_main[1])
axBR = fig.add_subplot(gs_main[3])

# sin_drift (shifted up)
name_head = 'sin_variance_inverse_'
result = load_pickle(output_path + name_head + '0_summarized')
func_x = load_pickle(output_path + f"fx_func_{name_head}0")
ode_res = fig7_ode_results[name_head]
predicted_x_bar = ode_res['predicted_x_bar']
predicted_var = ode_res['predicted_var']
x_vals_display = np.linspace(-25, 800, num_x)
axTR.plot(x_vals_display, func_x(x_vals_display) + shift_fac, color='black')
axTR.plot(x_vals_display, -func_x(x_vals_display) + shift_fac, color='black')
T = result['T']
D = result['D']
j_range = [int(np.where(T * D == val)[0][0]) for val in (5, 20, 40, 60, 85)]

plot_particle_scatter(axTR, name_head, func_x, predicted_x_bar, predicted_var, j_range,
                      shift=shift_fac, time_label=r'$tD/\textrm{z}_o^2=$',
                      label_y_offset=10, label_y_time=8.8, first_label_x_offset=-120)

weighted_x_mean, weighted_var_mean = load_seed_averages(name_head)
axBR.plot(weighted_x_mean, weighted_var_mean, color='C0', linestyle='dotted')
axBR.plot(predicted_x_bar, predicted_var, color='C2', alpha=1)
x_vals2 = np.linspace(0, int(np.max(weighted_x_mean)), 500)
axBR.plot(x_vals2, sigma2_sin_drift(x_vals2), color=DC1, alpha=1, linestyle='dashed')

# sin_nodrift (shifted down)
name_head = 'sin_nodrift_variance_inverse_'
result = load_pickle(output_path + name_head + '0_summarized')
func_x = load_pickle(output_path + f"fx_func_{name_head}0")
ode_res = fig7_ode_results[name_head]
predicted_x_bar = ode_res['predicted_x_bar']
predicted_var = ode_res['predicted_var']
x_vals_display = np.linspace(-25, 800, num_x)
axTR.plot(x_vals_display, func_x(x_vals_display) - shift_fac, color='black')
axTR.plot(x_vals_display, -func_x(x_vals_display) - shift_fac, color='black')
T = result['T']
D = result['D']
j_range = [int(np.where(T * D == val)[0][0]) for val in (5, 20, 40, 60, 85)]

plot_particle_scatter(axTR, name_head, func_x, predicted_x_bar, predicted_var, j_range,
                      shift=-shift_fac)

axTR.tick_params(axis='both', direction='in', labelsize=22)
axTR.set_xlabel(r'$x^*$', fontsize=25)
axTR.set_ylabel(r'$y^*$', fontsize=25)
axTR.set_xlim([-50, 850])
axTR.set_ylim([-8.8, 8.8])
axTR.plot(np.linspace(-50, 850, 500), np.zeros(500), linestyle='dashed', color='gray')

weighted_x_mean, weighted_var_mean = load_seed_averages(name_head)
axBR.plot(weighted_x_mean, weighted_var_mean, color='C0', linestyle='dotted')
axBR.plot(predicted_x_bar, predicted_var, color='C2', alpha=1)
x_vals2 = np.linspace(0, int(np.max(weighted_x_mean)), 500)
axBR.plot(x_vals2, sigma2_sin_nodrift(x_vals2), color=DC1, alpha=1, linestyle='dashed')
axBR.set_xlabel(r'$\overline{x}^*$', fontsize=25)
axBR.set_ylabel(r'$\sigma^{*2}_x$', fontsize=25)
axBR.set_ylim([200, 600])
axBR.set_xlim([-50, 850])
axBR.tick_params(axis='both', direction='in', labelsize=22)

# Labels and annotations
axTL.text(-0.12, 1.01, r'$(a)$', transform=axTL.transAxes, fontsize=22, va='top', ha='left')
axTL.text(0.04, 0.96, r'$(i)$', transform=axTL.transAxes, fontsize=22, va='top', ha='left')
axTR.text(-0.12, 1.01, r'$(b)$', transform=axTR.transAxes, fontsize=22, va='top', ha='left')
axTR.text(0.04, 0.97, r'$(i)$', transform=axTR.transAxes, fontsize=22, va='top', ha='left')
axTR.text(0.105, 0.97, r'$\sigma^{*2}_{x, \mathrm{target}} = 0.3x^* + 50\sin(2 \pi x^*/600) + 300$',
          transform=axTR.transAxes, fontsize=20, va='top', ha='left')
axTR.text(0.04, 0.475, r'$\sigma^{*2}_{x, \mathrm{target}} = 50\sin(2 \pi x^*/600) + 300$',
          transform=axTR.transAxes, fontsize=20, va='top', ha='left')
axTL.text(0.12, 0.96, r'$\sigma^{*2}_{x, \mathrm{target}} = 300$',
          transform=axTL.transAxes, fontsize=22, va='top', ha='left')
axBL.text(0.01, 1.15, r'$(ii)$', transform=axBL.transAxes, fontsize=22, va='top', ha='left')
axBR.text(0.01, 1.15, r'$(ii)$', transform=axBR.transAxes, fontsize=22, va='top', ha='left')
fig.text(0.5, 0.98, r'$Pe_{\mathrm{z}} = 10$', ha='center', va='bottom', fontsize=25)
axTR.set_yticks([-shift_fac - 3, -shift_fac, -shift_fac + 3, shift_fac - 3, shift_fac, shift_fac + 3])
axTR.set_yticklabels([r'$-3$', r'$0$', r'$3$', r'$-3$', r'$0$', r'$3$'])
axBR.text(100, 420, r'$\sigma^{*2}_{x, \mathrm{ODE}}$' + ' (—)', color='C2', fontsize=22,
          horizontalalignment='center', verticalalignment='bottom')
axBR.text(370, 400, r'$\sigma^{*2}_{x, \mathrm{B}}$' + r' ($\cdot\cdot\cdot$)', color='C0', fontsize=22,
          horizontalalignment='center', verticalalignment='bottom')
axBR.text(480, 500, r'$\sigma^{*2}_{x, \mathrm{target}}$' + r' ($--$)', color=DC1, fontsize=22,
          horizontalalignment='center', verticalalignment='bottom')

plt.show()

### Figure S1 Visualization

In [ ]:
result = load_pickle(output_path + "result_dumbell_Pe0_5_L0_3_moment_initVar10_seed_0")
D = result['D']
fig, ax1 = plt.subplots(figsize=(12, 5))
a0_vals_dict = load_pickle(output_path + 'a0_dumbell_Pe0_5')
a0_vals = np.sqrt(a0_vals_dict['time_results'] * D)
H_vals = a0_vals_dict['H_values']
slope = -0.00075
x_values = (H_vals / 2 - 1) / slope
a0_interp = UnivariateSpline(x_values, a0_vals, s=0.2)
h_interp = UnivariateSpline(x_values, H_vals / 2)

x_vals = np.linspace(0, 1200, 100)
ax1.plot(x_vals, a0_interp(x_vals), label=r'$\mathrm{Brownian}$', color='C0', linestyle='dotted')
ax1.plot(x_vals, np.ones(100) * 6, label=r'$\mathrm{Heuristic}$', color='C2')

ax2 = ax1.twinx()
ax2.plot(x_vals, h_interp(x_vals), color=DC1, linestyle='dashed', label=r'$\mathrm{Half}$' + r'-' + r'$\mathrm{height}$')
ax2.spines['right'].set_color(DC1)
ax2.spines['left'].set_visible(False)
ax2.set_ylabel('$h(x^*)$', color=DC1, fontsize=28)
ax2.tick_params(axis='y', labelsize=22, direction='in', colors=DC1)
ax2.set_ylim([-0.1, 1.3])
ax2.set_yticks([0, 0.5, 1])

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, fontsize=22, loc='upper left', ncols=3, columnspacing=0.7)

ax1.set_ylabel(r'$a_o$', fontsize=28)
ax1.set_xlabel(r'$x^*$', fontsize=28)
ax1.tick_params(axis='both', labelsize=22, direction='in')

ax1.text(780, 7.9, r'$a_o = \sqrt{t_{\scriptscriptstyle \mathrm{diff}}D}$', color='C0', fontsize=28, va='top', ha='left')
ax1.text(100, 6.75, r'$a_o \approx 2l_o$', color='C2', fontsize=28, va='top', ha='left')
ax1.text(510, 7.5, r'$h(x^*)$', color=DC1, fontsize=28, va='top', ha='left')

plt.show()